## Data loader pipeline
Drives the `training.data_loader` package (B2.1) against a single AI4Arctic scene: load per-band normalisation stats, open the scene once, and iterate its chips. No inline NetCDF/resampling/GCP logic here — only calls into the modularized package in `src/training`.

In [ ]:
import logging
from pathlib import Path

from training import ALL_BANDS, load_band_means, load_scene, yield_chips

logging.basicConfig(level=logging.INFO)

### Configuration
S3 location of the precomputed per-band statistics (the B1 artefact) and the scene to process.

In [ ]:
BUCKET = "prescient-ice-data"
STATS_KEY = "training_data/ai4arctic/statistics/dataset_stats.json"
AWS_PROFILE = "spk_data"

SCENE_PATH = Path(
    "../../S1A_EW_GRDM_1SDH_20180124T194759_20180124T194859_020301_022AA4_1F75"
    "_icechart_dmi_201801241950_SouthEast_RIC.nc"
)

### Load per-band normalisation statistics
Calls `training.load_band_means`, which fetches the B1 stats JSON from S3 and returns the per-band means used to fill land/nodata pixels.

In [ ]:
band_means = load_band_means(BUCKET, STATS_KEY, ALL_BANDS, profile=AWS_PROFILE)
print(f"Loaded stats for {len(band_means)} bands")

### Load the scene
`training.load_scene` opens the NetCDF exactly once, builds the valid-pixel mask, substitutes land/nodata pixels with the per-band mean, resamples AMSR2/ERA5/incidence-angle to SAR resolution, builds the GCP interpolators, and parses the ice-chart CT labels — all in one call.

In [ ]:
scene = load_scene(SCENE_PATH, band_means)

print(f"scene_id: {scene.scene_id}")
print(f"SAR shape: {scene.sar_h} x {scene.sar_w}")
print(f"Valid fraction: {scene.valid_mask.mean():.3f}")
print(f"Incidence angle shape: {scene.incidence_angle.shape}")
print(f"Time encoding: {scene.time_encoding}")

### Iterate chips
`training.yield_chips` tiles the in-memory scene into 256×256 chips on the regular grid (trailing chip backward-shifted to the scene edge), skipping fully-invalid chips and logging how many were skipped. No further file access happens here — everything comes from the `SceneArrays` built above.

In [ ]:
chips = list(yield_chips(scene))
print(f"Chips yielded: {len(chips)}")

### Validate chip output
Sanity-check shapes on the first chip and confirm the last chip's trailing edge reaches the scene boundary.

In [ ]:
first, last = chips[0], chips[-1]

assert first.sar.shape == (2, 256, 256)
assert first.amsr2.shape == (14, 256, 256)
assert first.era5.shape == (6, 256, 256)
assert first.incidence_angle.shape == (256, 256)
assert first.time_encoding.shape == (4,)
assert first.latlon_encoding.shape == (4,)
assert all(chip.valid_mask.any() for chip in chips)

assert last.chip_row_start + 256 == scene.sar_h
assert last.chip_col_start + 256 == scene.sar_w

print("Shape checks passed")
print(f"First chip: {first.chip_id}")
print(f"Last chip:  {last.chip_id}")